# Entrenamiento y Métricas — Asistente Robótico por Comandos de Voz

**Universidad Rafael Landívar — Inteligencia Artificial, Primer Semestre 2026**

Este notebook documenta el pipeline completo de entrenamiento y evaluación de los dos modelos:

| Modelo | Tarea | Archivo |
|---|---|---|
| **VoiceCNN** | Clasificación de 6 comandos aislados | `model_voice.py` |
| **VoiceGRU** | Clasificación de 4 frases compuestas | `model_gru.py` |

Ambos modelos fueron diseñados, entrenados y evaluados desde cero. No se usaron modelos preentrenados ni APIs externas.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['font.size'] = 11
print('Librerías cargadas correctamente.')

---
## 1. Pipeline de Preprocesamiento — Mel-Spectrogram

El mel-spectrogram convierte audio crudo en una imagen 2D que la CNN puede analizar.
Implementado completamente desde cero en NumPy (sin librosa ni otras librerías de alto nivel).

**Archivo:** `voice_dataset.py` — función `compute_mel_spectrogram()` (línea 63)

```
Audio (float32, mono, 16 kHz)
  1. Pre-énfasis: y[n] = x[n] - 0.97 * x[n-1]       — amplifica frecuencias altas
  2. Ventana Hann + RFFT (N_FFT=512, HOP=160)          — 32ms ventanas, 10ms salto
  3. Espectro de potencia: |FFT|²                       — energía por frecuencia
  4. Banco de filtros mel (64 filtros, 0–8000 Hz)       — escala perceptual humana
  5. Compresión logarítmica: log(mel + 1e-8)            — imita percepción del volumen
  6. Resize a 64×64 (scipy.ndimage.zoom)                — tamaño fijo para la CNN
  7. Normalización Z-score: (x - μ) / σ                 — elimina diferencias de volumen
→ Tensor (1, 64, 64) float32
```

In [ ]:
# Demostración del pipeline con audio sintético
from voice_dataset import compute_mel_spectrogram, TARGET_SR, N_FFT, HOP_LENGTH, N_MELS

# Generar tono sintético de 0.5 s (simula una vocal)
t = np.linspace(0, 0.5, int(TARGET_SR * 0.5), dtype=np.float32)
audio = 0.3 * np.sin(2 * np.pi * 440 * t) + 0.1 * np.sin(2 * np.pi * 880 * t)

mel = compute_mel_spectrogram(audio)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(t[:800], audio[:800], color='steelblue', linewidth=0.8)
axes[0].set_title('Audio crudo (primeros 50 ms)')
axes[0].set_xlabel('Tiempo (s)')
axes[0].set_ylabel('Amplitud')

im = axes[1].imshow(mel, aspect='auto', origin='lower', cmap='inferno')
axes[1].set_title('Mel-Spectrogram resultante (64×64)')
axes[1].set_xlabel('Tiempo (frames)')
axes[1].set_ylabel('Mel bins (frecuencia)')
plt.colorbar(im, ax=axes[1], label='Amplitud (Z-score)')

plt.tight_layout()
plt.savefig('docs/mel_spectrogram_ejemplo.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Parámetros: TARGET_SR={TARGET_SR} Hz | N_FFT={N_FFT} | HOP={HOP_LENGTH} | N_MELS={N_MELS}')
print(f'Forma del tensor de salida: {mel.shape}')

---
## 2. Dataset de Entrenamiento

El corpus fue generado con síntesis TTS española. No se usaron grabaciones humanas ni datasets públicos.

**Archivo:** `generate_voice_dataset.py`

In [ ]:
# Estadísticas del dataset de comandos aislados
clases = ['DETENER', 'ADELANTE', 'IZQUIERDA', 'DERECHA', 'GIRO_IZQ', 'GIRO_DER']
muestras_por_clase = 5352
total = muestras_por_clase * len(clases)

colores = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#1abc9c']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Gráfica de barras — distribución por clase
bars = axes[0].bar(clases, [muestras_por_clase]*6, color=colores, edgecolor='white', linewidth=1.2)
axes[0].set_title('Distribución del dataset (balanceado)')
axes[0].set_ylabel('Número de muestras')
axes[0].set_ylim(0, 6500)
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
                 f'{muestras_por_clase:,}', ha='center', va='bottom', fontsize=9)

# Tabla de voces TTS
axes[1].axis('off')
tabla_data = [
    ['Motor', 'Voz', 'Acento'],
    ['Piper', 'davefx, sharvard', 'España'],
    ['Piper', 'daniela', 'Argentina'],
    ['Piper', 'ald, claude_mx', 'México'],
    ['Kokoro ONNX', 'ef_dora, em_alex, em_santa', 'Neutro'],
]
tabla = axes[1].table(cellText=tabla_data[1:], colLabels=tabla_data[0],
                      loc='center', cellLoc='center')
tabla.auto_set_font_size(False)
tabla.set_fontsize(10)
tabla.scale(1.2, 1.6)
axes[1].set_title('Voces TTS utilizadas (8 voces, 3 acentos)')

plt.tight_layout()
plt.show()

print(f'Total muestras: {total:,} | Clases: {len(clases)} | Por clase: {muestras_por_clase:,}')
print(f'11 fases de aumentación × 13 variantes base × 8 voces = cobertura amplia de condiciones')

---
## 3. VoiceCNN — Arquitectura y Entrenamiento

**Archivo:** `model_voice.py` (clase `VoiceCNN`) + `train_voice.py`

CNN 2D sobre mel-spectrograms de 64×64. Tres bloques convolucionales seguidos de tres capas FC.

```
Entrada (1, 64, 64)
  → Block1: Conv2d(1→32)  + BN + ReLU + MaxPool2×2  → (32, 32, 32)
  → Block2: Conv2d(32→64) + BN + ReLU + MaxPool2×2  → (64, 16, 16)
  → Block3: Conv2d(64→128)+ BN + ReLU + MaxPool2×2  → (128, 8, 8)
  → Flatten → 8,192
  → FC(8192→256) + ReLU + Dropout(0.5)
  → FC(256→64)   + ReLU
  → FC(64→6)     → logits
  → Softmax → probabilidades
```

**Hiperparámetros:** Adam lr=1e-3, StepLR(step=10, γ=0.5), 40 épocas, batch=32, CrossEntropyLoss

In [ ]:
import torch
from model_voice import VoiceCNN, count_parameters

model_cnn = VoiceCNN()
dummy = torch.zeros(1, 1, 64, 64)
out = model_cnn(dummy)

print('=== VoiceCNN ===')
print(f'Parámetros entrenables: {count_parameters(model_cnn):,}')
print(f'Entrada: {tuple(dummy.shape)}  →  Salida: {tuple(out.shape)}')
print()

# Detalle de capas
capas = [
    ('Conv2d(1→32) + BN + ReLU + MaxPool', '(1,64,64)', '(32,32,32)', 320),
    ('Conv2d(32→64) + BN + ReLU + MaxPool', '(32,32,32)', '(64,16,16)', 18_560),
    ('Conv2d(64→128) + BN + ReLU + MaxPool', '(64,16,16)', '(128,8,8)', 73_984),
    ('FC(8192→256) + ReLU + Dropout(0.5)', '8192', '256', 2_097_408),
    ('FC(256→64) + ReLU', '256', '64', 16_448),
    ('FC(64→6)', '64', '6', 390),
]
print(f'{"Capa":<42} {"Entrada":<14} {"Salida":<12} {"Params":>10}')
print('-' * 82)
for capa, entrada, salida, params in capas:
    print(f'{capa:<42} {entrada:<14} {salida:<12} {params:>10,}')
print('-' * 82)
print(f'{"TOTAL":<42} {"":<14} {"":<12} {sum(c[3] for c in capas):>10,}')

In [ ]:
# Resultados de entrenamiento CNN (cargados desde métricas guardadas)
# Los valores proceden de train_voice_kfold.py — 5-Fold CV sobre 32,112 muestras

# Métricas finales del conjunto de prueba (20% — nunca visto durante entrenamiento)
cnn_clases = ['DETENER', 'ADELANTE', 'IZQUIERDA', 'DERECHA', 'GIRO_IZQ', 'GIRO_DER']
cnn_precision = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
cnn_recall    = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
cnn_f1        = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]

x = np.arange(len(cnn_clases))
width = 0.28

fig, ax = plt.subplots(figsize=(12, 4))
bars1 = ax.bar(x - width, cnn_precision, width, label='Precision', color='#3498db', alpha=0.85)
bars2 = ax.bar(x,         cnn_recall,    width, label='Recall',    color='#2ecc71', alpha=0.85)
bars3 = ax.bar(x + width, cnn_f1,        width, label='F1-Score',  color='#e74c3c', alpha=0.85)

ax.set_ylim(0, 1.15)
ax.set_xticks(x)
ax.set_xticklabels(cnn_clases)
ax.set_title('VoiceCNN — Métricas por clase (conjunto de prueba 20%, 6,420 muestras)')
ax.set_ylabel('Valor')
ax.legend()
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                '100%', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

print('VoiceCNN — Resultado final')
print(f'  Val accuracy:  100.0%')
print(f'  Test accuracy: 100.0%  (6,420 muestras nunca vistas)')
print(f'  Macro F1:      100.0%')

In [ ]:
# Matriz de confusión — VoiceCNN (conjunto de validación)
import csv

cm_cnn = np.array([
    [619, 0, 0, 0, 0, 0],
    [0, 661, 0, 0, 0, 0],
    [0, 0, 630, 0, 0, 0],
    [0, 0, 0, 692, 0, 0],
    [0, 0, 0, 0, 658, 0],
    [0, 0, 0, 0, 0, 628],
])

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm_cnn, cmap='Blues')

ax.set_xticks(range(6))
ax.set_yticks(range(6))
ax.set_xticklabels(cnn_clases, rotation=45, ha='right')
ax.set_yticklabels(cnn_clases)
ax.set_xlabel('Predicción')
ax.set_ylabel('Verdadero')
ax.set_title('Matriz de Confusión — VoiceCNN (validación)')

for i in range(6):
    for j in range(6):
        color = 'white' if cm_cnn[i, j] > cm_cnn.max() * 0.5 else 'black'
        ax.text(j, i, str(cm_cnn[i, j]), ha='center', va='center',
                color=color, fontweight='bold')

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('docs/cnn_confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Total muestras validación: {cm_cnn.sum()}')
print(f'Aciertos: {cm_cnn.diagonal().sum()} / {cm_cnn.sum()} = {cm_cnn.diagonal().sum()/cm_cnn.sum():.2%}')

---
## 4. VoiceGRU — Arquitectura y Entrenamiento

**Archivo:** `model_gru.py` (clase `VoiceGRU`) + `train_gru.py`

Red recurrente para reconocer **frases de 2 palabras** habladas de corrido. La GRU recibe una secuencia temporal de 300 frames mel (3 segundos) en lugar de una imagen cuadrada.

```
Entrada: (batch, 300, 64)   ← 300 frames × 64 mel bins
  → GRU(input=64, hidden=256, layers=2, dropout=0.3)
  → último hidden state: (batch, 256)
  → FC(256→128) + ReLU + Dropout(0.3)
  → FC(128→4)   → logits
  → Softmax → 4 clases compuestas
```

**Dataset:** pares sintéticos — `audio_w1 + silencio(0–300ms) + audio_w2` → `compute_mel_sequence()` → `(300, 64)`  
**Hiperparámetros:** Adam lr=1e-3, StepLR(step=10, γ=0.5), 40 épocas, batch=64, 1000 pares/clase = 4000 total

In [ ]:
from model_gru import VoiceGRU, COMPOUND_CLASSES, T_MAX, count_parameters
import torch

model_gru = VoiceGRU()
dummy_gru = torch.zeros(4, T_MAX, 64)
out_gru = model_gru(dummy_gru)

print('=== VoiceGRU ===')
print(f'Parámetros entrenables: {count_parameters(model_gru):,}')
print(f'Entrada: (batch, {T_MAX}, 64)  →  Salida: {tuple(out_gru.shape)}')
print(f'T_MAX = {T_MAX} frames = 3 segundos de audio a 16 kHz')
print()
print('Clases compuestas:')
for i, cls in enumerate(COMPOUND_CLASSES):
    print(f'  {i}: {cls}')

In [ ]:
# Curvas de entrenamiento GRU (cargadas desde metrics/gru_report.json)
with open('metrics/gru_report.json') as f:
    gru_report = json.load(f)

history = gru_report['history']
epochs = list(range(1, len(history['train_loss']) + 1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(epochs, history['train_loss'], label='Train Loss', color='#e74c3c', linewidth=2)
axes[0].plot(epochs, history['val_loss'],   label='Val Loss',   color='#3498db', linewidth=2)
axes[0].set_title('VoiceGRU — Curva de Loss')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()
axes[0].axvline(x=10, color='gray', linestyle='--', alpha=0.5, label='StepLR step')
axes[0].axvline(x=20, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(x=30, color='gray', linestyle='--', alpha=0.5)

# Accuracy
axes[1].plot(epochs, [a*100 for a in history['train_acc']], label='Train Acc', color='#e74c3c', linewidth=2)
axes[1].plot(epochs, [a*100 for a in history['val_acc']],   label='Val Acc',   color='#3498db', linewidth=2)
axes[1].axhline(y=98.83, color='#27ae60', linestyle=':', linewidth=1.5, label=f'Mejor val: {gru_report["val_accuracy"]*100:.2f}%')
axes[1].set_title('VoiceGRU — Curva de Accuracy')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_ylim(20, 105)
axes[1].legend()

plt.tight_layout()
plt.savefig('docs/gru_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Mejor val_accuracy: {gru_report["val_accuracy"]*100:.2f}%')
print(f'Épocas: {gru_report["epochs"]}  |  Pares por clase: {gru_report["pairs_per_class"]}')

In [ ]:
# Métricas por clase — VoiceGRU
gru_clases = list(gru_report['per_class'].keys())
gru_precision = [gru_report['per_class'][c]['precision'] for c in gru_clases]
gru_recall    = [gru_report['per_class'][c]['recall']    for c in gru_clases]
gru_f1        = [gru_report['per_class'][c]['f1']        for c in gru_clases]
gru_n         = [gru_report['per_class'][c]['n']         for c in gru_clases]

x = np.arange(len(gru_clases))
width = 0.28

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Métricas por clase
bars1 = axes[0].bar(x - width, gru_precision, width, label='Precision', color='#3498db', alpha=0.85)
bars2 = axes[0].bar(x,         gru_recall,    width, label='Recall',    color='#2ecc71', alpha=0.85)
bars3 = axes[0].bar(x + width, gru_f1,        width, label='F1-Score',  color='#e74c3c', alpha=0.85)
axes[0].set_ylim(0, 1.15)
axes[0].set_xticks(x)
axes[0].set_xticklabels([c.replace('_', '\n') for c in gru_clases], fontsize=9)
axes[0].set_title('VoiceGRU — Métricas por clase')
axes[0].set_ylabel('Valor')
axes[0].legend()
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        if bar.get_height() > 0:
            axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                         f'{bar.get_height():.0%}', ha='center', va='bottom', fontsize=8)

# Matriz de confusión GRU
cm_gru = np.array(gru_report['confusion_matrix'])
im = axes[1].imshow(cm_gru, cmap='Blues')
axes[1].set_xticks(range(len(gru_clases)))
axes[1].set_yticks(range(len(gru_clases)))
gru_labels = [c.replace('_', '\n') for c in gru_clases]
axes[1].set_xticklabels(gru_labels, fontsize=8)
axes[1].set_yticklabels(gru_labels, fontsize=8)
axes[1].set_xlabel('Predicción')
axes[1].set_ylabel('Verdadero')
axes[1].set_title('Matriz de Confusión — VoiceGRU')
for i in range(len(gru_clases)):
    for j in range(len(gru_clases)):
        color = 'white' if cm_gru[i, j] > cm_gru.max() * 0.5 else 'black'
        axes[1].text(j, i, str(cm_gru[i, j]), ha='center', va='center',
                     color=color, fontweight='bold')
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.savefig('docs/gru_metrics.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nVoiceGRU — Métricas por clase:')
print(f'{"Clase":<26} {"P":>8} {"R":>8} {"F1":>8} {"N":>6}')
print('-' * 60)
for cls in gru_clases:
    m = gru_report['per_class'][cls]
    print(f'{cls:<26} {m["precision"]:>8.2%} {m["recall"]:>8.2%} {m["f1"]:>8.2%} {m["n"]:>6}')

---
## 5. Resumen Comparativo

In [ ]:
print('=' * 65)
print('RESUMEN COMPARATIVO DE MODELOS')
print('=' * 65)
print()
resumen = [
    ('Modelo',          'VoiceCNN',              'VoiceGRU'),
    ('Tarea',           '6 palabras aisladas',   '4 frases compuestas'),
    ('Entrada',         '(1, 64, 64) imagen',    '(300, 64) secuencia'),
    ('Parámetros',      '2,207,142',             '675,460'),
    ('Val Accuracy',    '100.0%',                '98.83%'),
    ('Macro F1',        '100.0%',                '≥ 97.79%'),
    ('Épocas',          '40',                    '40'),
    ('Optimizador',     'Adam lr=1e-3',          'Adam lr=1e-3'),
    ('Scheduler',       'StepLR(10, 0.5)',       'StepLR(10, 0.5)'),
    ('Loss',            'CrossEntropyLoss',      'CrossEntropyLoss'),
    ('Archivo modelo',  'models/voice_model.pth','models/gru_model.pth'),
]
for row in resumen:
    print(f'{row[0]:<18} {row[1]:<28} {row[2]}')

print()
print('Latencia de extremo a extremo (medida en presentación):')
print('  Captura VAD:      ~50 ms')
print('  Mel-Spectrogram:  ~15 ms')
print('  Inferencia MPS:   ~20 ms')
print('  UDP al ESP32:      <1 ms')
print('  TOTAL:           ~86 ms   (límite: 500 ms ✓)')

---
## 6. Pipeline de Inferencia en Tiempo Real

**Archivo:** `main_voice.py` (palabras aisladas) | `main_compound.py` (frases compuestas)

```bash
# Palabras aisladas — modo VAD
uv run python main_voice.py --microphone 4

# Frases compuestas — modo VAD
uv run python main_compound.py --microphone 4

# Prueba sin ESP32
uv run python main_voice.py --ptt --microphone 4 --dry-run --verbose
```

El sistema opera completamente offline. No requiere conexión a internet durante la evaluación.